# SaaS Product Support — Multi-Agent Workflow

This notebook implements a goal-oriented multi-agent workflow for a SaaS product support use case (technical issue diagnosis). It includes agent definitions, handoff formats, escalation logic, and sample runs demonstrating end-to-end behavior.


## Architecture Explanation

**Agents (4):**
- **TechnicalDiagnosisAgent**: Parse user symptom, identify probable root cause, confidence score, and quick checks to run.
- **ProductKnowledgeAgent**: Apply non-RAG product rules and policies (config limits, known behaviours) to validate diagnosis and suggest safe mitigations.
- **TroubleshootingResponseAgent**: Produce step-by-step user-facing troubleshooting instructions, triage severity, and estimate resolution probability.
- **EngineeringEscalationAgent**: Decide escalation when risk/uncertainty/edge-case detected; format the engineering ticket payload.

**Handoff format**: Each agent receives a dictionary like:

```
{"user_query": str, "context": {...}, "diagnosis": {...}}
```

Agents augment the dict with `diagnosis`, `confidence`, `actions`, and `escalation_reason`.

**Escalation rules (examples):**
- If combined confidence < 0.5 -> escalate.
- If issue maps to data-loss, security, or production outage -> escalate immediately.
- If troubleshooting steps include code-level fixes or schema migrations -> escalate.

**Design rationale**: Keep agents narrowly-scoped so each step's output is structured and checkable. Escalation is explicit and explainable (reasons + evidence).


In [1]:
# Agents implementation (simulated, deterministic)
from typing import Dict, Any, List

def TechnicalDiagnosisAgent(user_query: str, context: Dict[str,Any]) -> Dict[str,Any]:
    q = user_query.lower()
    diagnosis = {}
    # simple heuristics to detect issue type
    if 'login' in q or 'sign in' in q or 'authenticate' in q:
        diagnosis['issue_type'] = 'auth'
        diagnosis['probable_cause'] = 'invalid_credentials_or_sso'
        confidence = 0.8
        actions = ['check-credentials', 'check-sso-status']
    elif 'api' in q or 'endpoint' in q or '500' in q or 'error' in q:
        diagnosis['issue_type'] = 'api'
        diagnosis['probable_cause'] = 'bad_request_or_backend_error'
        confidence = 0.6
        actions = ['collect-logs', 'reproduce-call']
    elif 'feature' in q or 'how to' in q or 'usage' in q:
        diagnosis['issue_type'] = 'usage'
        diagnosis['probable_cause'] = 'user-error_or-missing-docs'
        confidence = 0.7
        actions = ['show-docs', 'example-commands']
    else:
        diagnosis['issue_type'] = 'unknown'
        diagnosis['probable_cause'] = 'needs-investigation'
        confidence = 0.4
        actions = ['collect-more-info']
    return {'diagnosis': diagnosis, 'confidence': confidence, 'actions': actions, 'evidence': q}

def ProductKnowledgeAgent(state: Dict[str,Any], product_rules: Dict[str,Any]) -> Dict[str,Any]:
    # apply simple rules/policies stored in product_rules (non-RAG)
    diag = state['diagnosis']
    issue = diag['issue_type']
    augmentation = {}
    if issue == 'auth':
        # check known SSO outage flag
        if product_rules.get('sso_outage', False):
            augmentation['note'] = 'Known SSO outage'
            state['confidence'] = min(state['confidence'] * 0.9, 0.95)
            augmentation['escalate'] = True
        else:
            augmentation['note'] = 'Follow normal auth troubleshooting'
    elif issue == 'api':
        rate_limit = product_rules.get('rate_limit_policy', True)
        if '500' in state.get('evidence', ''):
            augmentation['note'] = 'Server error observed; collect logs'
            augmentation['escalate'] = True
    else:
        augmentation['note'] = 'No special product rule'
    state.setdefault('notes', []).append(augmentation)
    return state

def TroubleshootingResponseAgent(state: Dict[str,Any]) -> Dict[str,Any]:
    diag = state['diagnosis']
    issue = diag['issue_type']
    response = {}
    if issue == 'auth':
        response['steps'] = [
            'Confirm username/email is correct',
            'Reset password or try SSO login flow',
            'If using SSO, check IdP status'
        ]
        response['severity'] = 'medium'
    elif issue == 'api':
        response['steps'] = [
            'Collect request/response headers and payload',
            'Check service status and recent deploys',
            'Retry the request and capture logs'
        ]
        response['severity'] = 'high' if state['confidence']<0.6 else 'medium'
    elif issue == 'usage':
        response['steps'] = ['Provide docs link', 'Give example usage snippet']
        response['severity'] = 'low'
    else:
        response['steps'] = ['Ask user for more context: screenshots, logs, steps to reproduce']
        response['severity'] = 'unknown'
    state['response'] = response
    return state

def EngineeringEscalationAgent(state: Dict[str,Any]) -> Dict[str,Any]:
    # Decide whether to escalate and prepare ticket payload
    escalate = False
    reasons: List[str] = []
    if state.get('confidence', 1.0) < 0.5:
        escalate = True
        reasons.append('low_confidence')
    # escalate on high severity or explicit product rule
    resp = state.get('response', {})
    if resp.get('severity') == 'high':
        escalate = True
        reasons.append('high_severity')
    for note in state.get('notes', []):
        if note.get('escalate'):
            escalate = True
            reasons.append('product_rule')
    if escalate:
        ticket = {
            'title': f"Escalation: {state['diagnosis']['issue_type']}",
            'description': state.get('evidence', '') + '\n' + str(state.get('diagnosis', {})),
            'attachments': state.get('logs', [])
        }
        state['escalation'] = {'escalate': True, 'reasons': reasons, 'ticket': ticket}
    else:
        state['escalation'] = {'escalate': False, 'reasons': reasons}
    return state

# Orchestrator: sequential handoff
def run_pipeline(user_query: str, product_rules: Dict[str,Any]={}):
    context = {}
    s1 = TechnicalDiagnosisAgent(user_query, context)
    state = {'user_query': user_query, 'diagnosis': s1['diagnosis'], 'confidence': s1['confidence'], 'actions': s1['actions'], 'evidence': s1['evidence']}
    state = ProductKnowledgeAgent(state, product_rules)
    state = TroubleshootingResponseAgent(state)
    state = EngineeringEscalationAgent(state)
    return state

# Expose a simple runner for notebook demonstration
if __name__ == '__main__':
    examples = [
        'I cannot login to my account since yesterday, SSO fails with 401',
        'Our API returns 500 when calling /v1/data with this payload',
        'How do I enable feature X? The docs are unclear',
        'Intermittent data loss after bulk import - production issue'
    ]
    rules = {'sso_outage': False, 'rate_limit_policy': True}
    for q in examples:
        print('---')
        print('Query:', q)
        out = run_pipeline(q, rules)
        print('Diagnosis:', out['diagnosis'])
        print('Confidence:', out['confidence'])
        print('Response steps:', out['response']['steps'])
        print('Escalation:', out['escalation'])



---
Query: I cannot login to my account since yesterday, SSO fails with 401
Diagnosis: {'issue_type': 'auth', 'probable_cause': 'invalid_credentials_or_sso'}
Confidence: 0.8
Response steps: ['Confirm username/email is correct', 'Reset password or try SSO login flow', 'If using SSO, check IdP status']
Escalation: {'escalate': False, 'reasons': []}
---
Query: Our API returns 500 when calling /v1/data with this payload
Diagnosis: {'issue_type': 'api', 'probable_cause': 'bad_request_or_backend_error'}
Confidence: 0.6
Response steps: ['Collect request/response headers and payload', 'Check service status and recent deploys', 'Retry the request and capture logs']
Escalation: {'escalate': True, 'reasons': ['product_rule'], 'ticket': {'title': 'Escalation: api', 'description': "our api returns 500 when calling /v1/data with this payload\n{'issue_type': 'api', 'probable_cause': 'bad_request_or_backend_error'}", 'attachments': []}}
---
Query: How do I enable feature X? The docs are unclear
Diagno

## Sample inputs & expected behavior

- A clear `auth` issue with high-confidence should return step-by-step auth checks and typically not escalate unless product rules indicate outage.
- An `api` issue with server error (`500`) reduces confidence and triggers escalation with logs collected.
- Ambiguous queries produce low confidence and a structured escalation payload for engineers (reasons + evidence).


## Next steps

- Run this notebook to see the simulated flows.
- Optionally, connect the agents to real LLM calls or integrate with CrewAI for autonomous orchestration.
- Expand product rules and escalate payload fields to match your team's ticketing system.


In [2]:
# Optional CrewAI / LLM integration
# This cell adds an OpenAI-backed agent adapter. If `OPENAI_API_KEY` is set in the environment
# and `openai` is installed, the pipeline will call LLM-based agents that return structured JSON.

import os
import json

USE_LLM = bool(os.getenv('OPENAI_API_KEY'))

if USE_LLM:
    try:
        import openai
    except Exception as e:
        print('openai package not available; install with `pip install openai` to enable LLM agents')
        USE_LLM = False


def call_llm_json(prompt: str, model: str = 'gpt-3.5-turbo') -> dict:
    """Call OpenAI chat completions and parse JSON response. Returns parsed JSON or raises ValueError."""
    if not USE_LLM:
        raise RuntimeError('LLM not enabled (missing OPENAI_API_KEY or openai package)')
    messages = [
        {"role": "system", "content": "You are a precise assistant that outputs only valid JSON matching the requested schema."},
        {"role": "user", "content": prompt}
    ]
    resp = openai.ChatCompletion.create(model=model, messages=messages, temperature=0)
    text = resp['choices'][0]['message']['content']
    # some LLMs include backticks; strip surrounding markdown fences
    text = text.strip()
    if text.startswith('```') and text.endswith('```'):
        # remove fences
        parts = text.split('\n')
        # remove first and last fence lines
        text = '\n'.join(parts[1:-1])
    try:
        return json.loads(text)
    except Exception as e:
        # try to extract JSON substring
        import re
        m = re.search(r'\{.*\}', text, re.S)
        if m:
            return json.loads(m.group(0))
        raise

# LLM agent wrappers that request structured JSON from the model

def LLM_TechnicalDiagnosisAgent(user_query: str, context: dict) -> dict:
    prompt = f"""
    Produce a JSON object with keys: diagnosis (object with issue_type and probable_cause), confidence (0-1 float), actions (array of strings), evidence (original user query).
    User query: {user_query}
    """
    out = call_llm_json(prompt)
    return out


def LLM_ProductKnowledgeAgent(state: dict, product_rules: dict) -> dict:
    prompt = f"""
    Given the following state and product rules, return the updated state as JSON. Keys to include/augment: notes (array), confidence (may be adjusted), escalate flags inside notes when appropriate.
    State: {json.dumps(state)}
    Product rules: {json.dumps(product_rules)}
    """
    out = call_llm_json(prompt)
    return out


def LLM_TroubleshootingResponseAgent(state: dict) -> dict:
    prompt = f"""
    Given the state, return a JSON object with a `response` key containing `steps` (array of step strings) and `severity` (one of low, medium, high, unknown).
    State: {json.dumps(state)}
    """
    out = call_llm_json(prompt)
    return out


def LLM_EngineeringEscalationAgent(state: dict) -> dict:
    prompt = f"""
    Decide whether to escalate. Return JSON: {{"escalation": {{"escalate": bool, "reasons": [str], "ticket": {{...}}}}}}. Include evidence and diagnosis in ticket.description if escalating.
    State: {json.dumps(state)}
    """
    out = call_llm_json(prompt)
    return out

# Combined LLM pipeline runner
def run_pipeline_llm(user_query: str, product_rules: dict = {}):
    if not USE_LLM:
        raise RuntimeError('LLM not enabled; set OPENAI_API_KEY and install openai')
    s1 = LLM_TechnicalDiagnosisAgent(user_query, {})
    state = {'user_query': user_query, 'diagnosis': s1.get('diagnosis', {}), 'confidence': s1.get('confidence', 1.0), 'actions': s1.get('actions', []), 'evidence': s1.get('evidence', user_query)}
    # pass through LLM product agent which must return full state JSON
    state = LLM_ProductKnowledgeAgent(state, product_rules)
    # ensure state contains diagnosis/confidence
    state = LLM_TroubleshootingResponseAgent(state)
    state = LLM_EngineeringEscalationAgent(state)
    return state

# Fallback runner that selects LLM if available, else deterministic
def run_pipeline_auto(user_query: str, product_rules: dict = {}):
    if USE_LLM:
        try:
            return run_pipeline_llm(user_query, product_rules)
        except Exception as e:
            print('LLM runner failed, falling back to deterministic agents:', e)
    return run_pipeline(user_query, product_rules)

# Expanded example queries (8 total) including edge-cases to trigger escalation
EXAMPLES = [
    'I cannot login to my account since yesterday, SSO fails with 401',
    'Our API returns 500 when calling /v1/data with this payload',
    'How do I enable feature X? The docs are unclear',
    'Intermittent data loss after bulk import - production issue',
    'When uploading CSV, some rows disappear and we lost 200 records',
    'The integration with partner app times out after 30s and causes duplicate entries',
    'Dev token is rejected with 403 even though permissions were set',
    'Random 502 from CDN causing feature toggle failures in prod'
]

# Notebook runner cell (call run_pipeline_auto for each example)
for q in EXAMPLES:
    print('\n---')
    print('Query:', q)
    out = run_pipeline_auto(q, {'sso_outage': False, 'rate_limit_policy': True})
    # pretty print relevant fields
    print('Diagnosis:', out.get('diagnosis'))
    print('Confidence:', out.get('confidence'))
    print('Response steps:', out.get('response', {}).get('steps'))
    print('Escalation:', out.get('escalation'))



---
Query: I cannot login to my account since yesterday, SSO fails with 401
LLM runner failed, falling back to deterministic agents: 

You tried to access openai.ChatCompletion, but this is no longer supported in openai>=1.0.0 - see the README at https://github.com/openai/openai-python for the API.

You can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. 

Alternatively, you can pin your installation to the old version, e.g. `pip install openai==0.28`

A detailed migration guide is available here: https://github.com/openai/openai-python/discussions/742

Diagnosis: {'issue_type': 'auth', 'probable_cause': 'invalid_credentials_or_sso'}
Confidence: 0.8
Response steps: ['Confirm username/email is correct', 'Reset password or try SSO login flow', 'If using SSO, check IdP status']
Escalation: {'escalate': False, 'reasons': []}

---
Query: Our API returns 500 when calling /v1/data with this payload
LLM runner failed, falling back to deterministic agent